# Lección 11 - Protocolo de Contexto de Modelo (MCP)

El **Protocolo de Contexto de Modelo (MCP)** es un estándar abierto que permite a los agentes descubrir y usar dinámicamente herramientas, recursos y fuentes de datos en tiempo real. En lugar de codificar herramientas directamente en un agente, MCP permite que los agentes se conecten a servidores externos que exponen capacidades bajo demanda.

En esta lección, aprenderás:
- Qué es MCP y por qué es importante para los sistemas de agentes
- Cómo funciona la arquitectura cliente-servidor de MCP
- Cómo construir agentes que usen el descubrimiento de herramientas al estilo MCP


## Configuración

**Requisitos previos:**
- Proyecto Azure AI Foundry con un modelo desplegado
- Ejecutar `az login` para autenticación


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ¿Qué es el Protocolo de Contexto del Modelo (MCP)?

MCP define una forma estándar para que los agentes de IA descubran e interactúen con herramientas externas y fuentes de datos:

- **Servidor MCP**: Expone herramientas, recursos y prompts a través de un protocolo estándar
- **Cliente MCP**: El entorno de ejecución del agente que se conecta a los servidores y descubre las capacidades disponibles
- **Descubrimiento Dinámico**: Los agentes no necesitan herramientas codificadas; descubren lo que está disponible en tiempo de ejecución

Esto es poderoso para construir sistemas de agentes extensibles donde se pueden agregar nuevas capacidades sin modificar el código del agente.


## Cómo funciona MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. El agente (cliente MCP) se conecta a un servidor MCP
2. El servidor responde con una lista de herramientas disponibles y sus esquemas
3. El agente puede entonces llamar a cualquier herramienta descubierta durante su razonamiento
4. Los resultados fluyen de vuelta a través del mismo protocolo


## Simulando el Descubrimiento de Herramientas MCP

Dado que un servidor MCP real requiere un proceso de servidor en ejecución, demostraremos el patrón utilizando funciones `@tool` que simulan lo que un servicio de alojamiento conectado a MCP proporcionaría.

En producción, estas herramientas se descubrirían dinámicamente desde un servidor MCP en lugar de definirse localmente.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Construyendo un Agente con Herramientas al Estilo MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP en Producción

En un entorno de producción, MCP habilita patrones poderosos:

- **Descubrimiento dinámico de herramientas**: Los agentes se conectan a servidores MCP y descubren herramientas en tiempo de ejecución
- **Arquitectura desacoplada**: Los proveedores de herramientas pueden actualizarse independientemente del agente
- **Compartición entre organizaciones**: Los equipos pueden exponer capacidades a través de servidores MCP que cualquier agente puede usar
- **Soporte del Microsoft Agent Framework**: MAF tiene soporte integrado para clientes MCP a través de la integración `mcp`

Para usar un servidor MCP real con MAF, te conectarías mediante `hosted_mcp_tool()` o la integración del cliente MCP.

**Aprende más:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Resumen

En esta lección, aprendiste:
- **MCP** es un estándar abierto para el descubrimiento dinámico de herramientas entre agentes y proveedores de herramientas
- La **arquitectura cliente-servidor** permite que los agentes descubran capacidades en tiempo de ejecución
- MCP posibilita **sistemas de agentes extensibles y desacoplados** donde se pueden agregar herramientas sin cambios en el código
- Microsoft Agent Framework proporciona **soporte MCP incorporado** para uso en producción


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso legal**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos hacemos responsables de malentendidos o interpretaciones erróneas derivadas del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
